# Session 3: from the photo back to the flame
**A. Real-world problem.** You have a photo of a flame and want the glow of its inside, not
the sum along each line of sight. If the flame is round about its axis, one photo is enough:
peel it like an onion. The catch is noise. Every real photo has a little noise, and the
inversion multiplies it, most of all near the axis. You will measure that amplification,
fight it with a smoothing method, and see the price (bias). Everything is checked against
the exact pair Gaussian -> Gaussian from Session 2.

In [ ]:
import pathlib

import numpy as np

OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"
R_MM = 4.0

## Step 1: the forward model and the exact truth
The matrix W (Session 2) turns ring emissions into a brightness profile. To avoid the
"inverse crime" (testing a method on data made by the very same approximation), the test
photos below come from a *fine* numerical integral, not from W.

In [ ]:
def ring_matrix(N, R=R_MM):
    dr = R / N
    edges = dr * np.arange(N + 1)
    y = edges[:N]
    inner = np.sqrt(np.clip(edges[None, :-1] ** 2 - y[:, None] ** 2, 0.0, None))
    outer = np.sqrt(np.clip(edges[None, 1:] ** 2 - y[:, None] ** 2, 0.0, None))
    return 2.0 * (outer - inner)


def radii(N, R=R_MM):
    """Ring middles r_j (mm)."""
    return (R / N) * (np.arange(N) + 0.5)


def gauss_eps(r, a):
    return np.exp(-(np.asarray(r, dtype=float) / a) ** 2)


def ring_eps(r, r0, w):
    return np.exp(-((np.asarray(r, dtype=float) - r0) / w) ** 2)


def photo_profile(eps_fn, N, R=R_MM, n_fine=3000):
    """Brightness at y_i = i*dr from a fine integral (not from W): I = 2 * int eps(sqrt(y^2+u^2)) du."""
    I = np.zeros(N)
    for i, y in enumerate((R / N) * np.arange(N)):
        u = np.linspace(0.0, np.sqrt(R * R - y * y), n_fine + 1)
        f = eps_fn(np.sqrt(y * y + u * u))
        I[i] = 2.0 * np.sum(0.5 * (f[1:] + f[:-1]) * np.diff(u))
    return I


def add_noise(I, frac, rng):
    """Gaussian noise with standard deviation frac * (peak brightness)."""
    return I + frac * I.max() * rng.standard_normal(I.shape)

## Step 2: onion peeling
W is upper triangular: the outermost line of sight (i = N-1) crosses only the outermost ring.
Solve it first, then move inwards, each time subtracting what the outer rings already explain:

    eps_i = ( I_i - sum over j > i of W_ij eps_j ) / W_ii

In [ ]:
def onion_peel(I, R=R_MM):
    I = np.asarray(I, dtype=float)
    N = len(I)
    W = ring_matrix(N, R)
    eps = np.zeros(N)
    for i in range(N - 1, -1, -1):
        eps[i] = (I[i] - W[i, i + 1:] @ eps[i + 1:]) / W[i, i]
    return eps

## Step 3: noise amplification
Inversion is linear: eps = W^-1 I. If each I_i has independent noise of size sigma, the noise
in ring i is sigma times the length of row i of W^-1. This number is the *gain* of ring i.

In [ ]:
def noise_gain(N, R=R_MM):
    """Standard deviation of eps_i divided by the noise sigma of the photo, for onion peeling."""
    return np.linalg.norm(np.linalg.inv(ring_matrix(N, R)), axis=1)

## Step 4: smoothing (Tikhonov regularisation)
Instead of demanding W eps = I exactly, minimise  |W eps - I|^2 + lam * s * |D eps|^2,
where D takes second differences (it is zero for a straight-line profile and large for a
jagged one) and s = mean(diag(W^T W)) makes `lam` a pure number. Large lam = smooth but
biased; lam = 0 gives noisy onion peeling.

In [ ]:
def smooth_invert(I, lam, R=R_MM):
    I = np.asarray(I, dtype=float)
    N = len(I)
    W = ring_matrix(N, R)
    D = np.diff(np.eye(N), n=2, axis=0)
    s = np.trace(W.T @ W) / N
    return np.linalg.solve(W.T @ W + lam * s * (D.T @ D), W.T @ I)


def trial(eps_fn, N, lam, frac, n_trials=100, seed=0, R=R_MM):
    """Monte Carlo: return (mean estimate, std of estimate, truth) over noisy photos."""
    rng = np.random.default_rng(seed)
    I0 = photo_profile(eps_fn, N, R)
    est = np.array([smooth_invert(add_noise(I0, frac, rng), lam, R) for _ in range(n_trials)])
    return est.mean(axis=0), est.std(axis=0), eps_fn(radii(N, R))


def bias_noise_total(eps_fn, N, lam, frac, n_trials=100, seed=0, R=R_MM):
    """RMS bias, RMS noise and RMS total error (all relative to the peak of eps)."""
    mean, std, truth = trial(eps_fn, N, lam, frac, n_trials, seed, R)
    peak = truth.max()
    bias = np.sqrt(np.mean((mean - truth) ** 2)) / peak
    noise = np.sqrt(np.mean(std ** 2)) / peak
    return float(bias), float(noise), float(np.hypot(bias, noise))


def bias_noise_exact(eps_fn, N, lam, frac, R=R_MM):
    """The same three numbers without random draws. The estimate is linear, eps_hat = G I, so
    bias = G I_true - eps_true and noise_i = sigma * (length of row i of G); sigma = frac * peak(I)."""
    W = ring_matrix(N, R)
    D = np.diff(np.eye(N), n=2, axis=0)
    s = np.trace(W.T @ W) / N
    G = np.linalg.solve(W.T @ W + lam * s * (D.T @ D), W.T)
    I0 = photo_profile(eps_fn, N, R)
    truth = eps_fn(radii(N, R))
    peak = truth.max()
    bias = np.sqrt(np.mean((G @ I0 - truth) ** 2)) / peak
    noise = frac * I0.max() * np.sqrt(np.mean(np.sum(G * G, axis=1))) / peak
    return float(bias), float(noise), float(np.hypot(bias, noise))

## Step 5: mini-project material, a candle-like flame
A synthetic side-view "photo" of an axisymmetric flame, in mm: the flame is hollow near the
base (a ring of glow around dark wax vapour), fills in higher up, and tapers to a tip.
`candle_emission(r, z)` is the hidden truth; `candle_photo` is what the camera records
(both sides of the axis, optional noise).

In [ ]:
def candle_emission(r, z):
    s = 3.2 * (z / 6.0) * np.exp(1.0 - z / 6.0)          # flame radius envelope (mm)
    r0 = 0.7 * s * max(0.0, 1.0 - z / 10.0)              # radius of the glow ring (mm); 0 above z = 10 mm
    e = np.exp(-(z / 22.0) ** 2)                         # overall strength fades with height
    return e * np.exp(-((np.asarray(r, dtype=float) - r0) / (0.55 * s)) ** 2)


def candle_photo(N=40, heights=range(1, 21), frac=0.0, seed=1, R=6.0):
    """Rows = heights (mm), columns = offsets y_i (mm) on one side of the axis (flame is symmetric)."""
    rng = np.random.default_rng(seed)
    rows = [photo_profile(lambda r, z=z: candle_emission(r, z), N, R) for z in heights]
    img = np.array(rows)
    if frac > 0:
        img = img + frac * img.max() * rng.standard_normal(img.shape)
    return img


def rel_rms(est, truth):
    """RMS error divided by the peak of the truth."""
    return float(np.sqrt(np.mean((est - truth) ** 2)) / np.max(np.abs(truth)))


def invert_row(row, lam, R=6.0):
    """Radial emission from one image row: onion peeling if lam == 0, otherwise smoothed."""
    return onion_peel(row, R) if lam == 0 else smooth_invert(row, lam, R)


def worst_row_error(img, lam, heights=(4, 8, 12), R=6.0):
    """Largest relative RMS error over the chosen heights (mm); img rows start at z = 1 mm."""
    rr = radii(img.shape[1], R)
    return max(rel_rms(invert_row(img[z - 1], lam, R), candle_emission(rr, z)) for z in heights)


LAM_GRID = (0.0, 0.01, 0.03, 0.1, 0.3, 1.0, 3.0, 10.0)

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    N, a, frac = 40, 1.0, 0.01
    r = radii(N)
    gauss = lambda q: gauss_eps(q, a)
    ringf = lambda q: ring_eps(q, 1.5, 0.4)
    rng = np.random.default_rng(3)

    # exact pair: noise-free onion peeling of a Gaussian
    I0 = photo_profile(gauss, N)
    e0 = onion_peel(I0)
    print(f"noise-free onion peeling, Gaussian a=1 mm, N=40: worst error {np.max(np.abs(e0 - gauss(r))):.4f} of peak 1.0")
    gain = noise_gain(N)
    print(f"noise gain (eps noise / photo noise): axis ring {gain[0]:.1f}, middle ring {gain[N // 2]:.1f}, outer ring {gain[-1]:.2f}")

    # one noisy photo
    In = add_noise(I0, frac, rng)
    print(f"noisy photo (1% of peak): onion peeling RMS error {rel_rms(onion_peel(In), gauss(r)):.3f} of the peak, "
          f"smoothed (lam=0.1) {rel_rms(smooth_invert(In, 0.1), gauss(r)):.3f}")

    # bias-noise trade-off
    lams = np.logspace(-6, 1, 29)
    stats = np.array([bias_noise_exact(gauss, N, l, frac) for l in lams])
    best = lams[np.argmin(stats[:, 2])]
    print(f"best smoothing for the Gaussian at 1% noise: lam = {best:.2g}, total error {stats[:, 2].min():.3f} of the peak")
    statr = np.array([bias_noise_exact(ringf, N, l, frac) for l in lams])
    bestr = lams[np.argmin(statr[:, 2])]
    print(f"best smoothing for the ring flame at 1% noise: lam = {bestr:.2g}, total error {statr[:, 2].min():.3f} of the peak")

    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    ax = axes[0]
    ax.semilogy(r, gain, "o-", ms=3)
    ax.set_xlabel("distance from the axis r (mm)")
    ax.set_ylabel("noise gain (eps noise / photo noise)")
    ax.set_title("Onion peeling amplifies noise near the axis")
    ax.grid(alpha=0.3, which="both")
    ax = axes[1]
    ax.plot(r, gauss(r), "k", lw=2, label="truth")
    ax.plot(r, onion_peel(In), ".-", label="onion peeling")
    ax.plot(r, smooth_invert(In, best), "-", lw=2, label=f"smoothed, lam = {best:.2g}")
    ax.set_xlabel("distance from the axis r (mm)")
    ax.set_ylabel("emission eps (a.u. per mm)")
    ax.set_title("Gaussian flame, 1% noise")
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
    ax = axes[2]
    ax.loglog(lams, stats[:, 0], label="bias")
    ax.loglog(lams, stats[:, 1], label="noise")
    ax.loglog(lams, stats[:, 2], "k", lw=2, label="total")
    ax.set_xlabel("smoothing strength lam (no unit)")
    ax.set_ylabel("RMS error / peak")
    ax.set_title("Smoothing trades noise for bias")
    ax.grid(alpha=0.3, which="both")
    ax.legend()
    fig.tight_layout()
    fig.savefig(OUT / "noise_and_smoothing.png", dpi=110)

    # ring-shaped flame: too much smoothing fills the hole
    fig2, axes2 = plt.subplots(1, 2, figsize=(10, 4))
    Ir = add_noise(photo_profile(ringf, N), frac, np.random.default_rng(4))
    ax = axes2[0]
    ax.plot(r, ringf(r), "k", lw=2, label="truth")
    for lam, col in ((1e-4, "tab:red"), (bestr, "tab:green"), (30.0, "tab:purple")):
        ax.plot(r, smooth_invert(Ir, lam), color=col, label=f"lam = {lam:.2g}")
    ax.set_xlabel("distance from the axis r (mm)")
    ax.set_ylabel("emission eps (a.u. per mm)")
    ax.set_title("Ring-shaped flame, 1% noise")
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
    ax = axes2[1]
    ax.loglog(lams, statr[:, 0], label="bias")
    ax.loglog(lams, statr[:, 1], label="noise")
    ax.loglog(lams, statr[:, 2], "k", lw=2, label="total")
    ax.set_xlabel("smoothing strength lam (no unit)")
    ax.set_ylabel("RMS error / peak")
    ax.set_title("Ring flame needs less smoothing")
    ax.grid(alpha=0.3, which="both")
    ax.legend()
    fig2.tight_layout()
    fig2.savefig(OUT / "ring_flame.png", dpi=110)

    # mini-project
    img = candle_photo(N=40, frac=0.03)
    zs = [4, 8, 12]
    R6 = 6.0
    rr = radii(40, R6)
    fig3, axes3 = plt.subplots(1, 4, figsize=(15, 3.8))
    axes3[0].imshow(np.hstack([img[::-1, ::-1], img[::-1]]), extent=[-6, 6, 1, 20], origin="upper", cmap="inferno", aspect="auto")
    axes3[0].set_title("Side-view photo (3% noise)")
    axes3[0].set_xlabel("offset y (mm)")
    axes3[0].set_ylabel("height z (mm)")
    for k, z in enumerate(zs):
        ax = axes3[k + 1]
        row = z - 1
        truth = candle_emission(rr, z)
        ax.plot(rr, truth, "k", lw=2, label="truth")
        ax.plot(rr, smooth_invert(img[row], 3.0, R6), label="smoothed lam=3")
        ax.plot(rr, onion_peel(img[row], R6), ":", label="onion peeling")
        ax.set_ylim(-0.5, 1.4)
        ax.set_title(f"height z = {z} mm")
        ax.set_xlabel("radius r (mm)")
        ax.set_ylabel("emission (a.u. per mm)")
        ax.grid(alpha=0.3)
        ax.legend(fontsize=7)
    fig3.tight_layout()
    fig3.savefig(OUT / "candle_rows.png", dpi=110)
    for lam in LAM_GRID:
        print(f"lam = {lam:5.2f}: worst-row error {worst_row_error(img, lam):.3f} of the peak")

## Where this stops
We invert one row at a time and assume a perfectly round, steady flame whose axis is known.
A puffing, tilted or noisy flame breaks those assumptions. That is the road to the Target:
see `promo/roadmap.md`.

## Coding tasks
1. **Easy.** Use `smooth_invert` on a noisy photo of a Gaussian with lam = 0.01, 0.3 and 10 and describe, in two sentences each, what you see.
2. **Stretch.** Change the number of rings in `noise_gain` from 40 to 20 and 80 and describe how the gain at the axis ring changes; explain it with the pixel size.
3. **Explain.** Explain why averaging the left and right halves of a real photo reduces the noise by about a factor sqrt(2), and why this is only allowed if the axis position is known.

## Assessment
- **A. Conceptual.** Why does the noise in the recovered emission grow towards the axis, and why does smoothing help but not for free?
- **B. Numerical.** For N = 40 rings the noise gain of the axis ring is 5.9 per mm. The photo noise is sigma = 0.02 a.u. What is the noise in the emission of the axis ring? Compare with a Gaussian flame whose peak emission is 1 a.u. per mm.
- **C. Graph and simulation interpretation.** On the trade-off graph of the Gaussian flame at 1 percent noise, how do the bias, noise and total change as lam goes from 0.00001 to 30? Where is the minimum of the total?
- **D. Design and optimisation challenge.** Design: a ring-shaped flame (r0 = 1.5 mm, w = 0.4 mm) is photographed with 1 percent noise and 40 rings. Choose between lam = 0.01, 0.1 and 10 and give the total error of each (use `bias_noise_exact`). What risk does the largest lam bring for a hollow flame?

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    # worked example: three rings, R = 3 mm, measured I = (12, 9.121, 4.472)
    got = L.onion_peel(np.array([12.0, 9.121, 4.472]), 3.0)
    checks.append((f"worked example: onion peeling of (12, 9.121, 4.472) gives eps = (3, 2, 1) (got {np.round(got, 2)})",
                   np.allclose(got, [3.0, 2.0, 1.0], atol=0.01)))
    W = L.ring_matrix(3, 3.0)
    checks.append(("worked example: forward model of eps = (3, 2, 1) gives I = (12, 9.121, 4.472)",
                   np.allclose(W @ np.array([3.0, 2.0, 1.0]), [12.0, 9.1206, 4.4721], atol=1e-3)))
    # exact pair, no noise
    N = 40
    r = L.radii(N)
    truth = L.gauss_eps(r, 1.0)
    e0 = L.onion_peel(L.photo_profile(lambda q: L.gauss_eps(q, 1.0), N))
    err = float(np.max(np.abs(e0 - truth)))
    checks.append((f"exact pair: noise-free onion peeling recovers the Gaussian to within 0.02 of its peak (worst {err:.3f})", err < 0.02))
    checks.append(("edge case: an all-zero photo gives zero emission", np.allclose(L.onion_peel(np.zeros(10)), 0.0) and np.allclose(L.smooth_invert(np.zeros(10), 0.5), 0.0)))
    I0 = L.photo_profile(lambda q: L.gauss_eps(q, 1.0), 20)
    checks.append(("lam = 0 smoothing equals onion peeling", np.allclose(L.smooth_invert(I0, 0.0), L.onion_peel(I0), atol=1e-5)))
    # noise gain
    g = L.noise_gain(N)
    checks.append((f"noise gain is largest at the axis: {g[0]:.1f} per mm there against {g[-1]:.2f} per mm at the edge", g[0] > 5 * g[-1] and g[0] > 5.0))
    rng = np.random.default_rng(0)
    Winv = np.linalg.inv(L.ring_matrix(N))
    sim = np.std(np.array([Winv @ rng.standard_normal(N) for _ in range(4000)]), axis=0)
    checks.append(("measured noise in ring 0 matches the predicted gain within 10%", abs(sim[0] / g[0] - 1) < 0.10))
    # bias / noise trade-off
    gauss = lambda q: L.gauss_eps(q, 1.0)
    lo = L.bias_noise_total(gauss, 40, 1e-4, 0.01, 60)
    mid = L.bias_noise_total(gauss, 40, 0.5, 0.01, 60)
    hi = L.bias_noise_total(gauss, 40, 30.0, 0.01, 60)
    checks.append(("more smoothing: bias rises (%.3f < %.3f < %.3f)" % (lo[0], mid[0], hi[0]), lo[0] < mid[0] < hi[0]))
    checks.append(("more smoothing: noise falls (%.3f > %.3f > %.3f)" % (lo[1], mid[1], hi[1]), lo[1] > mid[1] > hi[1]))
    checks.append(("there is a best smoothing in between: total error is smallest at lam = 0.5", mid[2] < lo[2] and mid[2] < hi[2]))
    checks.append((f"smoothing helps: 1% noise total error {lo[2]:.3f} (lam ~ 0) falls to {mid[2]:.3f}", mid[2] < lo[2] / 2))
    ex = L.bias_noise_exact(gauss, 40, 0.5, 0.01)
    checks.append(("the exact (no random draws) bias and noise agree with the Monte Carlo values within 15%",
                   abs(ex[0] / mid[0] - 1) < 0.15 and abs(ex[1] / mid[1] - 1) < 0.15))
    # design check: ring flame, 1% noise, three smoothing strengths
    ringg = lambda q: L.ring_eps(q, 1.5, 0.4)
    tot = [L.bias_noise_exact(ringg, 40, lam, 0.01)[2] for lam in (0.01, 0.1, 10.0)]
    checks.append(("design: ring flame at 1% noise, total error for lam = 0.01, 0.1, 10 is about 2.7%, 2.0%, 7.7% (best: 0.1)",
                   [round(100 * t, 1) for t in tot] == [2.7, 2.0, 7.7]))
    tot3 = [L.bias_noise_exact(gauss, 40, lam, 0.03)[2] for lam in (0.316, 1.0, 3.16)]
    checks.append(("design: Gaussian at 3% noise stays below 2.5% total error for lam >= 1 but not at lam = 0.3",
                   tot3[0] > 0.025 > max(tot3[1], tot3[2])))
    # limit case: over-smoothing fills the hole of a ring-shaped flame
    ringf = lambda q: L.ring_eps(q, 1.5, 0.4)
    Ir = L.photo_profile(ringf, 40)
    hole_ok = float(L.smooth_invert(Ir, 1e-4)[0])
    hole_bad = float(L.smooth_invert(Ir, 30.0)[0])
    checks.append((f"limit: heavy smoothing corrupts the dark hole of a ring flame (axis value {hole_ok:.2f} -> {hole_bad:.2f}, negative light is unphysical)", abs(hole_ok) < 0.05 and abs(hole_bad) > 0.2))
    # mini-project
    img = L.candle_photo(N=40, frac=0.03)
    worst = {lam: L.worst_row_error(img, lam) for lam in L.LAM_GRID}
    best = min(worst, key=worst.get)
    checks.append((f"mini-project: onion peeling on the 3% noise photo has a worst-row error above 0.25 (got {worst[0.0]:.3f})", worst[0.0] > 0.25))
    checks.append((f"mini-project: smallest worst-row error is at lam = 3 and is below 0.05 (got lam = {best}, {worst[best]:.3f})", best == 3.0 and worst[best] < 0.05))
    checks.append(("mini-project: the candle is hollow at z = 4 mm and filled at z = 12 mm",
                   float(L.candle_emission(0.0, 4)) < 0.6 * float(L.candle_emission(1.2, 4)) and float(L.candle_emission(0.0, 12)) == max(L.candle_emission(np.linspace(0, 5, 100), 12))))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
